In [3]:
# Importation des outils
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

# 1. Chargement des données
diabete = load_diabetes()
X = diabete.data
y = diabete.target

# 2. Découpage train/test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Affichage pour vérification
print(f"Taille de X (total) : {X.shape}")
print(f"Taille de X_train : {X_train.shape}")
print(f"Taille de X_test : {X_test.shape}")

Taille de X (total) : (442, 10)
Taille de X_train : (353, 10)
Taille de X_test : (89, 10)


In [4]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1. On initialise notre modèle
modele_baseline = LinearRegression()

# 2. On l'entraîne UNIQUEMENT sur les données d'entraînement (le cours)
modele_baseline.fit(X_train, y_train)

# 3. On lui demande de prédire les résultats sur les données de test (l'examen)
predictions = modele_baseline.predict(X_test)

# 4. On calcule nos métriques pour évaluer son travail
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"RMSE (Erreur moyenne) : {rmse:.2f}")
print(f"R² (Score) : {r2:.2f}")

RMSE (Erreur moyenne) : 53.85
R² (Score) : 0.45


In [5]:
from sklearn.model_selection import cross_val_score

# 1. On lance la validation croisée sur le jeu d'entraînement
# cv=5 signifie "5 plis" (5 découpages)
scores_mse_negatifs = cross_val_score(
    modele_baseline, 
    X_train, 
    y_train, 
    cv=5, 
    scoring='neg_mean_squared_error'
)

# 2. On repasse les scores en positif et on calcule la racine carrée (RMSE)
scores_rmse_cv = np.sqrt(-scores_mse_negatifs)

# 3. On calcule la moyenne et l'écart-type de ces 5 scores
moyenne_rmse_cv = scores_rmse_cv.mean()
ecart_type_rmse_cv = scores_rmse_cv.std()

print("Les 5 scores RMSE :", np.round(scores_rmse_cv, 2))
print(f"RMSE Moyenne (Cross-Validation) : {moyenne_rmse_cv:.2f}")
print(f"Écart-type de la RMSE : {ecart_type_rmse_cv:.2f}")

Les 5 scores RMSE : [52.53 58.94 52.09 56.5  59.81]
RMSE Moyenne (Cross-Validation) : 55.97
Écart-type de la RMSE : 3.18


In [6]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# On va tester ces 3 niveaux de complexité
degres = [1, 2, 4]

for degre in degres:
    # 1. On crée le pipeline : Transformation polynomiale -> Régression linéaire
    modele_poly = make_pipeline(PolynomialFeatures(degre), LinearRegression())
    
    # 2. On entraîne le modèle sur les données d'entraînement
    modele_poly.fit(X_train, y_train)
    
    # 3. On lui fait faire des prédictions sur le Train ET sur le Test
    pred_train = modele_poly.predict(X_train)
    pred_test = modele_poly.predict(X_test)
    
    # 4. On calcule l'erreur (RMSE) pour les deux
    rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))
    rmse_test = np.sqrt(mean_squared_error(y_test, pred_test))
    
    print(f"--- Degré {degre} ---")
    print(f"RMSE Train : {rmse_train:.2f}")
    print(f"RMSE Test  : {rmse_test:.2f}\n")

--- Degré 1 ---
RMSE Train : 53.56
RMSE Test  : 53.85

--- Degré 2 ---
RMSE Train : 48.92
RMSE Test  : 55.64

--- Degré 4 ---
RMSE Train : 0.00
RMSE Test  : 383.29



In [7]:
from sklearn.linear_model import Ridge, Lasso

# On garde le degré 4 pour observer la correction du sur-apprentissage
degre = 4

# On prépare un dictionnaire avec nos trois variantes pour les comparer facilement
modeles_reg = {
    "Régression Classique (Sans Régularisation)": LinearRegression(),
    "Régression Ridge (Alpha=1.0)": Ridge(alpha=1.0),
    "Régression Lasso (Alpha=1.0)": Lasso(alpha=1.0, max_iter=10000)
}

for nom, modele in modeles_reg.items():
    # 1. Création du pipeline avec le modèle spécifique
    pipeline = make_pipeline(PolynomialFeatures(degre), modele)
    
    # 2. Entraînement
    pipeline.fit(X_train, y_train)
    
    # 3. Prédiction
    pred_train = pipeline.predict(X_train)
    pred_test = pipeline.predict(X_test)
    
    # 4. Calcul des scores
    rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))
    rmse_test = np.sqrt(mean_squared_error(y_test, pred_test))
    
    print(f"--- {nom} ---")
    print(f"RMSE Train : {rmse_train:.2f}")
    print(f"RMSE Test  : {rmse_test:.2f}\n")

--- Régression Classique (Sans Régularisation) ---
RMSE Train : 0.00
RMSE Test  : 383.29

--- Régression Ridge (Alpha=1.0) ---
RMSE Train : 58.18
RMSE Test  : 55.45

--- Régression Lasso (Alpha=1.0) ---
RMSE Train : 62.13
RMSE Test  : 58.34



In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
import numpy as np

degre = 4

modeles_reg = {
    "Régression Classique": LinearRegression(),
    "Régression Ridge": Ridge(alpha=1.0),
    "Régression Lasso": Lasso(alpha=1.0, max_iter=10000)
}

for nom, modele in modeles_reg.items():
    # 1. Le pipeline complet : Polynôme -> Mise à l'échelle -> Modèle
    pipeline = make_pipeline(
        PolynomialFeatures(degre), 
        StandardScaler(), # <-- L'étape indispensable pour la régularisation !
        modele
    )
    
    # 2. Entraînement
    pipeline.fit(X_train, y_train)
    
    # 3. Prédiction
    pred_train = pipeline.predict(X_train)
    pred_test = pipeline.predict(X_test)
    
    # 4. Calcul des scores
    rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))
    rmse_test = np.sqrt(mean_squared_error(y_test, pred_test))
    
    print(f"--- {nom} ---")
    print(f"RMSE Train : {rmse_train:.2f}")
    print(f"RMSE Test  : {rmse_test:.2f}\n")

--- Régression Classique ---
RMSE Train : 0.00
RMSE Test  : 422.98

--- Régression Ridge ---
RMSE Train : 14.19
RMSE Test  : 201.07

--- Régression Lasso ---
RMSE Train : 42.99
RMSE Test  : 53.32

